# Visualize Augmentations & Data Generator

Inspect `JawKeypointSequenceDataset` and `ExperimentGroupedBatchSampler` before training.

Requires `../data/train.pkl` (run `Create Dataset/create_dataset.py` first).

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

sys.path.insert(0, str(Path.cwd()))
from dataset import JawKeypointSequenceDataset, get_train_transforms, keypoints_to_heatmaps
from sampler import ExperimentGroupedBatchSampler
from train import collate_batch

TRAIN_PKL = "../data/train.pkl"
WINDOW_SIZE = 8
IMG_H, IMG_W = 240, 320
BATCH_SIZE = 4
SEED = 42

In [ ]:
train_tf = get_train_transforms(IMG_H, IMG_W)
ds = JawKeypointSequenceDataset(
    TRAIN_PKL,
    transform=train_tf,
    window_size=WINDOW_SIZE,
    img_h=IMG_H,
    img_w=IMG_W,
)
sampler = ExperimentGroupedBatchSampler(
    ds.center_indices_for_sampler, BATCH_SIZE, seed=SEED
)
loader = DataLoader(ds, batch_sampler=sampler, collate_fn=collate_batch)
print(f"Dataset windows: {len(ds):,}")

In [ ]:
def denorm(img_t):
    """CHW tensor (ImageNet norm) → HWC uint8 for display."""
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    x = img_t.cpu().numpy().transpose(1, 2, 0)
    x = np.clip(x * std + mean, 0, 1)
    return (x * 255).astype(np.uint8)

x, hm, kps_orig, exp_ids, frames = next(iter(loader))
print("x:", tuple(x.shape))
print("heatmaps:", tuple(hm.shape))
print("keypoints_orig:", tuple(kps_orig.shape))
print("experiment_ids:", exp_ids.tolist())

In [ ]:
# 8-frame sequence grid for first sample (center frame highlighted)
sample_idx = 0
center = WINDOW_SIZE // 2
seq = x[sample_idx]

fig, axes = plt.subplots(1, WINDOW_SIZE, figsize=(2.5 * WINDOW_SIZE, 3))
for t, ax in enumerate(axes):
    ax.imshow(denorm(seq[t]))
    ax.set_title(f"t={t}" + (" *" if t == center else ""), fontsize=9)
    ax.axis("off")
plt.suptitle("Temporal window (center = keypoint supervision)")
plt.tight_layout()
plt.show()

In [ ]:
# Keypoints on center frame (green=tip, red=line)
center_img = denorm(seq[center])
tip = kps_orig[sample_idx, 0].numpy()
line = kps_orig[sample_idx, 1].numpy()
scale = np.array([IMG_W / 640, IMG_H / 480])

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.imshow(center_img)
ax.scatter(tip[0] * scale[0], tip[1] * scale[1], c="lime", s=60, label="tip (scaled)")
ax.scatter(line[0] * scale[0], line[1] * scale[1], c="red", s=60, label="line (scaled)")
ax.legend()
ax.set_title("Center frame keypoints (heatmap grid coords)")
ax.axis("off")
plt.show()

In [ ]:
# Augmentation comparison: same index, two loader passes → different aug, same transform across 8 frames
idx = 0
a = ds[idx]
b = ds[idx]
fig, axes = plt.subplots(2, WINDOW_SIZE, figsize=(2.5 * WINDOW_SIZE, 5))
for row, item in enumerate([a, b]):
    seq_t = item[0]
    for t in range(WINDOW_SIZE):
        axes[row, t].imshow(denorm(seq_t[t]))
        axes[row, t].axis("off")
    axes[row, 0].set_ylabel(f"draw {row+1}", rotation=0, labelpad=40)
plt.suptitle("Two augmentation draws for the same window index")
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap overlay on center frame
hm_sample = hm[sample_idx].numpy()
center_img = denorm(seq[center])

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(center_img)
axes[0].set_title("Center frame")
axes[0].axis("off")
axes[1].imshow(center_img)
axes[1].imshow(hm_sample[0], cmap="hot", alpha=0.5)
axes[1].set_title("Tip heatmap")
axes[1].axis("off")
axes[2].imshow(center_img)
axes[2].imshow(hm_sample[1], cmap="hot", alpha=0.5)
axes[2].set_title("Line heatmap")
axes[2].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Sampler check: each batch has a single experiment_id
for i, batch in enumerate(loader):
    exp_ids = batch[3].unique().tolist()
    print(f"batch {i}: experiment_ids={exp_ids}")
    if i >= 5:
        break

In [ ]:
# Within-experiment shuffle: epoch 0 vs epoch 1 first-batch indices differ, same experiment
def first_batch_indices(epoch):
    sampler.set_epoch(epoch)
    batch = next(iter(DataLoader(ds, batch_sampler=sampler, collate_fn=collate_batch)))
    return batch[3][0].item(), batch[4][:4].tolist()

e0 = first_batch_indices(0)
e1 = first_batch_indices(1)
print("epoch 0 — experiment_id, first frames:", e0)
print("epoch 1 — experiment_id, first frames:", e1)